did a buffer analysis and a bbx by distance, but i don't think the result is very much useful. 

In [ ]:
import ast
import hdbscan
import numpy as np
import pandas as pd
import seaborn as sns
import geopandas as gpd
from PIL import Image
import matplotlib.pyplot as plt
from shapely.geometry import Point
from sklearn.cluster import DBSCAN
import matplotlib.patches as mpatches
from shapely.geometry import MultiPoint
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score
import geopandas as gpd
import seaborn as sns
import matplotlib.pyplot as plt

# Set global display format to show up to 6 decimal places
# pd.options.display.float_format = '{:.6f}'.format

# 1. Load SVI waste Data

In [ ]:
SVI_csv_path = "/Users/wenlanzhang/Downloads/PhD_UCL/Data/Waste/img/Correct_SVI.csv"
df = pd.read_csv(SVI_csv_path)
df = df[(df['img_dir'] != 'ZWL/') & (df['img_dir'] != 'Faith/')]
df = df.drop_duplicates(subset=['lat', 'lon', 'img_name'], keep='first')
# df

# Ensure your df has 'lat' and 'lon' columns
df['geometry'] = df.apply(lambda row: Point(row['lon'], row['lat']), axis=1)

# Convert to a GeoDataFrame
gdf = gpd.GeoDataFrame(df, geometry='geometry', crs="EPSG:4326")
gdf

## Load Nairobi boundary shapefile

In [ ]:
# boundary = gpd.read_file("/Users/wenlanzhang/Downloads/PhD_UCL/Data/Waste/BaseMap/AoI_BigBoundary.gpkg")
boundary = gpd.read_file("/Users/wenlanzhang/Downloads/PhD_UCL/Data/Shp/NariobiShp/Shp_from_Constituency/Nairobi_shp_C.shp")

# Ensure CRS match (convert if necessary)
gdf = gdf.to_crs(boundary.crs)

# Clip points inside the boundary
waste_clipped_gdf = gpd.clip(gdf, boundary)

# Display result
waste_clipped_gdf

# 1个框外 + 35没有坐标
# Original count: 4082
# Clipped count: 4046
# Removed count: 36
# Cliped within Nairobi count: 3385

## Load Slum shapefile

In [ ]:
# Plot the points
slums_gdf = gpd.read_file("/Users/wenlanzhang/Downloads/PhD_UCL/Data/Waste/Angela/slumaps_nairobi_sett/slumaps_nairobi_sett.shp")

fig, ax = plt.subplots(figsize=(10, 8))
boundary.plot(ax=ax, color='none', edgecolor='black', linewidth=1, label="Boundary")
waste_clipped_gdf.plot(ax=ax, color='red', markersize=5, alpha=0.5)
slums_gdf.plot(ax=ax, color='lightgrey', edgecolor="lightgrey", alpha=0.5, label="Slum Areas")

plt.title("Fly-Tipping Locations")
plt.show()

In [ ]:
# Clip points inside the boundary
clipped_gdf = gpd.clip(gdf, boundary)
slums_gdf = gpd.clip(slums_gdf, boundary)

# Display result
clipped_gdf
# 3897->3236

slums_gdf
# 1998->1988

In [ ]:
# Reproject to UTM Zone 37S (EPSG:32737)
projected_crs = "EPSG:32737"  
boundary_projected = boundary.to_crs(projected_crs)
clipped_gdf_projected = clipped_gdf.to_crs(projected_crs)
slums_gdf_projected = slums_gdf.to_crs(projected_crs)

print(clipped_gdf_projected.crs)
print(boundary_projected.crs)
print(slums_gdf_projected.crs)

# Buffer

In [ ]:
# Create a 500m buffer around informal settlements
buffer_distance = 500  # in meters
slums_buffer_gdf = slums_gdf_projected.copy()
slums_buffer_gdf["geometry"] = slums_buffer_gdf.buffer(buffer_distance)

# 3. Spatial join: find waste points within the buffer
waste_within_buffer = gpd.sjoin(clipped_gdf_projected, slums_buffer_gdf, how="inner", predicate="intersects")

# Drop duplicates by geometry or by index of the original waste data
waste_within_buffer = waste_within_buffer.drop_duplicates(subset='geometry')  # OR use 'index_left'

# Check the corrected number
print(f"Unique waste points within {buffer_distance} meters of slums: {len(waste_within_buffer)}")

# Optional: save to file
# waste_within_buffer.to_file("waste_within_500m_of_slums.shp")

In [ ]:
len(waste_within_buffer)/len(clipped_gdf_projected)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

# Plot buffer zone
slums_buffer_gdf.plot(ax=ax, color='lightblue', alpha=0.3, edgecolor='blue', label="500m Buffer")

# Plot slum areas
slums_gdf_projected.plot(ax=ax, color='lightgrey', edgecolor="grey", alpha=0.6, label="Informal Settlements")

# Plot all waste points
clipped_gdf_projected.plot(ax=ax, color='red', markersize=5, alpha=0.4, label="All Fly-Tipping Locations")

# Highlight those within the buffer
waste_within_buffer.plot(ax=ax, color='green', markersize=5, alpha=0.8, label="Within Buffer")
boundary_projected.plot(ax=ax, color='none', edgecolor='black', linewidth=1, label="Boundary")

# Labels
plt.title("Fly-Tipping Locations Within 500m of Informal Settlements", fontsize=14)
# plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Define buffer distances
buffer_distances = [1, 5, 10, 30, 50, 80, 100, 250, 500, 800, 1000, 1250, 1500, 1750, 2000]

# Initialize new columns with 0
for dist in buffer_distances:
    clipped_gdf_projected[f'buffer_{dist}'] = 0

# Loop through each buffer and update columns
for dist in buffer_distances:
    slums_buffer = slums_gdf_projected.copy()
    slums_buffer["geometry"] = slums_buffer.buffer(dist)

    # Spatial join (returns waste points within buffer)
    joined = gpd.sjoin(clipped_gdf_projected, slums_buffer, how='inner', predicate='intersects')

    # Extract unique indices of matched waste points
    within_buffer_indices = joined.index.unique()

    # Update column
    clipped_gdf_projected.loc[within_buffer_indices, f'buffer_{dist}'] = 1

# Plot result
counts = [clipped_gdf_projected[f'buffer_{d}'].sum() for d in buffer_distances]

plt.figure(figsize=(10, 6))
plt.plot(buffer_distances, counts, marker='o')
plt.title("Number of Waste Points Within X Meters of Informal Settlements")
plt.xlabel("Buffer Distance (meters)")
plt.ylabel("Number of Waste Points")
plt.grid(True)
plt.xticks(buffer_distances)
plt.tight_layout()
plt.show()

In [ ]:
total_points = len(clipped_gdf_projected)
percentages = [100 * clipped_gdf_projected[f'buffer_{dist}'].sum() / total_points for dist in buffer_distances]

plt.plot(buffer_distances, percentages, marker='o')
plt.ylabel("Percentage of Waste Points")

In [ ]:
print(clipped_gdf_projected[[f'buffer_{d}' for d in buffer_distances]].sum())
# print(clipped_gdf_projected[[f'buffer_{d}' for d in buffer_distances]].value_counts())
# clipped_gdf_projected

In [ ]:
# Now compute distances correctly
clipped_gdf_projected["dist_to_slum"] = clipped_gdf_projected.geometry.apply(lambda pt: slums_gdf_projected.distance(pt).min())
clipped_gdf_projected

In [ ]:
clipped_gdf_projected['dist_to_slum'].describe()
clipped_gdf_projected[clipped_gdf_projected['dist_to_slum']<=1]

In [ ]:
clipped_gdf_projected[clipped_gdf_projected['dist_to_slum'] > 2000]

In [ ]:
# Ensure 'dist_to_slum' column is numeric
clipped_gdf_projected['dist_to_slum'] = pd.to_numeric(clipped_gdf_projected['dist_to_slum'], errors='coerce')

# Drop NaNs in 'dist_to_slum'
clipped_gdf_projected = clipped_gdf_projected.dropna(subset=['dist_to_slum'])

# Plot histogram
plt.figure(figsize=(10, 6))
plt.hist(clipped_gdf_projected['dist_to_slum'], bins=100, color='skyblue', edgecolor='black')
plt.title('Histogram of Distance to Slum')
plt.xlabel('Distance to Slum (meters)')
plt.ylabel('Frequency')
plt.xlim(0, 4000)
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Ensure 'dist_to_slum' column is numeric
clipped_gdf_projected['dist_to_slum'] = pd.to_numeric(clipped_gdf_projected['dist_to_slum'], errors='coerce')
clipped_gdf_projected['dist_to_slum_capped'] = clipped_gdf_projected['dist_to_slum'].clip(upper=2000)

# Drop NaNs in 'dist_to_slum'
clipped_gdf_projected = clipped_gdf_projected.dropna(subset=['dist_to_slum_capped'])

# Plot histogram
plt.figure(figsize=(10, 6))
plt.hist(clipped_gdf_projected['dist_to_slum_capped'], bins=50, color='skyblue', edgecolor='black')
plt.title('Histogram of Distance to Slum')
plt.xlabel('Distance to Slum (meters)')
plt.ylabel('Frequency')
plt.xlim(0, 2000)
plt.grid(True)
plt.tight_layout()
plt.show()

# Nearest Neighbor Ratio (NNR)

In [ ]:
# Compute the area in square meters
study_area = boundary_projected.geometry.area.sum() 

print(f"Total study area: {study_area} square meters")

In [ ]:
clipped_gdf_projected = waste_clipped_gdf.to_crs(projected_crs)
coords = np.array(list(zip(clipped_gdf_projected.geometry.x, clipped_gdf_projected.geometry.y)))

# Fit Nearest Neighbors Model
nbrs = NearestNeighbors(n_neighbors=2, algorithm='ball_tree').fit(coords)

# Find nearest distances
distances, _ = nbrs.kneighbors(coords)

# Compute mean nearest neighbor distance (excluding self-distance)
mean_dist = distances[:, 1].mean()

print(f"Mean Nearest Neighbor Distance: {mean_dist:.6f}")

In [ ]:
# Count number of fly-tipping points
N = len(clipped_gdf_projected)

# Compute expected mean NN distance under CSR
expected_dist_csr = 1 / (2 * np.sqrt(N / study_area))

print(f"Expected Mean NN Distance (CSR): {expected_dist_csr:.6f}")
print(f"Observed Mean NN Distance: {mean_dist:.6f}")

In [ ]:
# Compute Nearest Neighbor Ratio (NNR)
nnr = mean_dist / expected_dist_csr
print(f"Nearest Neighbor Ratio (NNR): {nnr:.8f}")

# Bounding box

In [ ]:
def parse_multiple_bboxes_full(bbox_str):
    IMG_WIDTH = 400
    IMG_HEIGHT = 300
    IMG_AREA = IMG_WIDTH * IMG_HEIGHT

    try:
        bbox_list = ast.literal_eval(bbox_str)

        if isinstance(bbox_list, list) and all(len(b) == 4 for b in bbox_list):
            widths, heights, areas = [], [], []

            for bbox in bbox_list:
                x_min, y_min, x_max, y_max = bbox
                width = x_max - x_min
                height = y_max - y_min
                area = width * height

                widths.append(width)
                heights.append(height)
                areas.append(area)

            bbox_count = len(areas)

            total_area = sum(areas)
            total_bbox_rel_area = total_area / IMG_AREA

            avg_area = total_area / bbox_count
            # avg_width = sum(widths) / bbox_count
            # avg_height = sum(heights) / bbox_count
            avg_bbox_rel_area = avg_area / IMG_AREA

            return pd.Series([
                bbox_count,
                total_area,
                total_bbox_rel_area,
                avg_area,
                # avg_width,
                # avg_height,
                avg_bbox_rel_area
            ])
    except Exception as e:
        print(f"Error parsing row: {e}")
    
    return pd.Series([0, None, None, None, None, None, None])


In [ ]:
clipped_gdf_projected

In [ ]:
clipped_gdf_projected[
    [
        'bbox_count',
        'total_bbox_area',
        'total_bbox_rel_area',
        'avg_bbox_area',
        # 'avg_bbox_width',
        # 'avg_bbox_height',
        'avg_bbox_rel_area'
    ]
] = clipped_gdf_projected['yolo_bbox'].apply(parse_multiple_bboxes_full)
clipped_gdf_projected

In [ ]:
plt.figure(figsize=(8, 6))
plt.hist(clipped_gdf_projected['avg_bbox_rel_area'].dropna(), bins=30, color='skyblue', edgecolor='black')
plt.title('Distribution of Bounding Box Relative Area')
plt.xlabel('Relative Area (bbox_area / image_area)')
plt.ylabel('Frequency')
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(6, 5))
sns.violinplot(y=clipped_gdf_projected['avg_bbox_rel_area'].dropna(), color='lightgreen')
plt.title('Violin Plot of Bounding Box Relative Area')
plt.ylabel('Relative Area')
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(6, 5))
sns.boxplot(y=clipped_gdf_projected['avg_bbox_rel_area'].dropna(), color='orange')
plt.title('Boxplot of Bounding Box Relative Area')
plt.ylabel('Relative Area')
plt.grid(True)
plt.show()


In [ ]:
# Load one image
image_path = "/Users/wenlanzhang/Downloads/PhD_UCL/Data/GoogleStreetView/Maoran/img/0/_/0_5dZzO8ljjqttHqSy-WbQ_0.jpg"
img = Image.open(image_path)
width, height = img.size

print(f"Image size: {width} x {height}")

# Group

In [ ]:
clipped_gdf_projected

In [ ]:
# Define distance bins and labels
bins = [-1, 500, float('inf')]
labels = ['Inside Slum –100m', '>2000m']

# Create a new column for distance group
clipped_gdf_projected['distance_group'] = pd.cut(
    clipped_gdf_projected['dist_to_slum'], bins=bins, labels=labels
)
group_stats = clipped_gdf_projected.groupby('distance_group', observed=True)['avg_bbox_rel_area'].describe()
print(group_stats)

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=clipped_gdf_projected, x='distance_group', y='avg_bbox_rel_area',     
            hue='distance_group', 
            legend=True,
            palette='Set2')
plt.title('Bounding Box Size (Relative Area) by Distance to Slum')
plt.xlabel('Distance to Slum')
plt.ylabel('Relative Area of BBox')
plt.grid(True)
plt.show()

In [ ]:
# Filter groups
group1 = clipped_gdf_projected[clipped_gdf_projected['distance_group'] == 'Inside Slum –100m']['avg_bbox_rel_area'].dropna()
group2 = clipped_gdf_projected[clipped_gdf_projected['distance_group'] == '>2000m']['avg_bbox_rel_area'].dropna()

# Plot
plt.figure(figsize=(10, 6))
plt.hist(group1, bins=30, alpha=0.5, label='0–100m', color='blue', edgecolor='black', density=True)
plt.hist(group2, bins=30, alpha=0.5, label='500–2000m', color='orange', edgecolor='black', density=True)

plt.title('Overlapping Histograms of BBox Relative Area by Distance to Slum')
plt.xlabel('Bounding Box Relative Area')
plt.ylabel('Density')
plt.legend()
plt.grid(True)
plt.show()